# 09 · Custom neuron models — basics

The current py-spinnaker2 repository already supports a variety of neuron models but you can of course extend them with custom implementations of arbitrary neuron models. This does not only include spiking models but any kind of computation in general, e.g., a rate-based model or an entirely different set of equations, as long as it fits into the basic framework. 

Within this tutorial, we will explore how to create a custom neuron model and what to look out for on the way. We will thereby focus on the general framework that py-spinnaker2 provides and what conditions your model must fulfill to fit it. More advanced and optional topics concerning custom neuron model creation for SpiNNaker2 will be covered in **custom neuron models, advanced** (coming soon).

**By the end you will be able to:**

- understand the different levels of neuron model implementation in the respective software repositories
- determine, whether a computational paradigm fits this framework
- identify and implement the relevant parts of a computational model 
- register & use your model in networks running on SpiNNaker2

> **Warning**
>
> **Running a custom neuron model requires a real SpiNNaker2 board.** The Brian2 backend implements only the built-in neuron models, so a model you register yourself cannot run on it — `brian2_sim` raises `Unsupported neuron model` for anything outside its built-in list.

## 1. Neuron Model Example - Adaptive Exponential IF

We will be using the Adaptive Exponential Integrate and Fire (short: *AdEx*) neuron model as an example throughout this tutorial. The AdEx model consists of two coupled differential equations, one for the membrane potential and one for the adaptation current, accounting for spike-frequency adaptation observed in real neurons. It is therefore a bit more complex than the regular LIF neuron. 

The membrane potential evolves according to:

$$
\begin{equation}
    C \frac{dV}{dt} = -g_L (V - E_L) + g_L \Delta_T \exp\!\left(\frac{V - V_T}{\Delta_T}\right) - w + I,
    \tag{1}
\end{equation}
$$

where $V$ is the membrane potential, $w$ the adaptation variable, $I$ the input current, $C$ the membrane capacitance, $g_L$ the leak conductance, $E_L$ the leak reversal potential, $V_T$ the threshold, and $\Delta_T$ the slope factor. The **exponential term** describes the process of spike generation and the upswing of the action potential. It drives the AdEx neuron's voltage sharply upward and accelerates on its own once it nears threshold. This mimics the fast rising edge of a real action potential rather than just crossing a fixed cutoff, as in, e.g., the LIF model. Since this runaway growth would mathematically continue to infinity, simulations simply stop and record a spike once the voltage passes a high cutoff value (e.g., 0 mV or +30 mV).

The **adaptation variable** $w$ reduces the membrane potential by an amount dynamically calculated by:

$$
\begin{equation}
    \tau_w \frac{dw}{dt} = a(V - E_L) - w
    \tag{2}
\end{equation}
$$

where $a$ is the adaptation coupling parameter and $\tau_w$ is the adaptation time constant. It acts like a slow hyperpolarizing current that opposes the input current, such that the neuron needs more net drive to reach the same $V_T$. This process effectively adapts the firing threshold for the neuron dynamically based on its recent subthreshold activity. 

After the neuron spiked, the membrane potential is immediately reset to a fixed reset potential $V_r$:

$$
    \text{at } t = t_f: \quad V \rightarrow V_r.
$$

The adaptation variable $w$ is changed at the same time by a constant $b$:

$$
    \text{at } t = t_f: \quad w \rightarrow w + b.
$$

This adds a second mechanism for threshold adaptation to the neuron model. While the previous one from Eq. 2 was based on the subthreshold activity of the neuron, this one updates the adaptation variable in response to the neuron's recent firing rate.

> **Note**
>
> Further information about the AdEx neuron model can be found either on [Scholarpedia](http://www.scholarpedia.org/article/Adaptive_exponential_integrate-and-fire_model) or in the original paper ([Brette and Gerstner, 2005](https://journals.physiology.org/doi/full/10.1152/jn.00686.2005)).

## 2. Process Overview

The overall process of creating a custom neuron model involves the creation of multiple C code files for defining how the model will be executed on the *chip-side* and a single Python source file with your neuron model for defining networks and specifying parameters on the *host-side*. The figure below summarizes the components involved in this process as well as their relationship to each other. The components and lines highlighted in red are the most relevant ones, particularly for this example. The synaptic parameters (`sw_spec`) and the addresses defined in the `lif_adex.py` file are mapped to the chip-side code with processes that are not relevant for the moment, hence their arrows terminate at the chip-side box and don't target a specific location within the chip-side. 

<div class="svg-theme-dark" style="display:block;">
  <img src="images/s2_neuron-model_overview_dark.svg" alt="Diagram">
</div>
<div class="svg-theme-light" style="display:none;">
  <img src="images/s2_neuron-model_overview_light.svg" alt="Diagram">
</div>

<style>
@media (prefers-color-scheme: light) {
  .svg-theme-dark  { display: none !important; }
  .svg-theme-light { display: block !important; }
}
body[data-jp-theme-light="true"]  .svg-theme-dark  { display: none !important; }
body[data-jp-theme-light="true"]  .svg-theme-light { display: block !important; }
body[data-jp-theme-light="false"] .svg-theme-dark  { display: block !important; }
body[data-jp-theme-light="false"] .svg-theme-light { display: none !important; }
body[data-md-color-scheme="default"] .svg-theme-dark  { display: none !important; }
body[data-md-color-scheme="default"] .svg-theme-light { display: block !important; }
body[data-md-color-scheme="slate"]   .svg-theme-dark  { display: block !important; }
body[data-md-color-scheme="slate"]   .svg-theme-light { display: none !important; }
</style>

While this can be a bit overwhelming at first glance, there are only a few functions and variables we actually have to touch for creating the AdEx neuron model, as shown in red above. The remaining parts can be inherited from the standard LIF neuron model. We will go through this process step-by-step in the next sections of this notebook, focusing particularly on these core components, which are sufficient for most neurons types.

The table below lists all files from the diagram above, along with a short description, their execution location (host-side/chip-side), and the amount of changes we need to perform compared to a regular LIF neuron. While we will need to create all of these files, we can simply copy most of them from the regular LIF neuron implementation (`lif_neuron`).


| File | Description | Exec. Location | Changes vs LIF |
| :- | :- | :- | :- |
| `lif_adex.py` | Contains the neuron model class in Python that is used to instantiate networks on the host side. | Host | **Major** |
| `main.c` | Initializes the neuronal data, routing tables, parameters, logging, etc. | Chip | None |
| `neuron.c/.h` | Implements the overall functionality of a neuronal population on a single PE (initialization, model update, resolve recordings). | Chip | None |
| `neuron_model_impl.c/.h` | Implements the computational logic of the neuronal state update (membrane potential). | Chip | **Major** |
| `synapses.c/.h` | Implements the processing functionality of a synapse at post-synaptic neurons. | Chip | None |
| `regions.h` | Defines the data specific region names for the respective sections in memory. | Chip | None |
| `global_params.h` | Defines the simulation wide parameters, shared by all neurons. | Chip | None |
| `param_defs.h` | Defines the neuronal parameters as a fallback, no need for modifications.  | Chip | None |
| `CMakeLists.txt` | Defines how to build the neuron. | Chip | None |





> **Note**
>
> For additional information on the implementation of the AdEx or other neuron models, please have a look at the [neuron model documentation](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/neuron-models.html).

#### Math to Code Variable Mapping for AdEx Model

The tables below describe the names for the *state variables* of the model and its *parameters* in the code that correspond to the mathematical variables from the equations above. For the AdEx model specifically, some names also differ between Python and C. We don't store all variables from the equations but use them (e.g., $g_L$) to pre-compute certain helper variables (e.g., `a_sub`), thereby increasing the overall speed of the simulation runs. The *biophysical parameters* from the equations are listed below together with the *pre-computed constants*

##### State variables (evolve every timestep)
| Math symbol | Meaning | C field | Location |
|---|---|---|---|
| $V(t)$ | Membrane potential | `V_membrane` | `neuron_t` |
| $w(t)$ | Adaptation current, **rescaled** to $w/g_L$ | `w_adapt` | `neuron_t` |
| — | Refractory countdown (no continuous-time equivalent) | `refract_timer` | `neuron_t` |

##### Direct pass-through parameters (same physical meaning, unit space, both sides)
| Math symbol | Meaning | Python param | C field | Units |
|---|---|---|---|---|
| $E_L$ | Leak/resting potential | `E_L` | `E_L` | mV |
| $V_T$ | Exponential soft threshold | `V_T` | `V_T` | mV |
| $\Delta_T$ | Slope factor | `delta_T` | `delta_T` | mV |
| $V_{reset}$ | Post-spike reset voltage | `V_reset` | `V_reset` | mV |
| $V_{peak}$ | Hard spike-detection threshold | `V_peak` | `threshold` *(renamed!)* | mV |
| $V_{init}$ | Initial membrane voltage | `V_init` | `V_init` | mV |
| $w_{init}$ | Initial adaptation variable | `w_init` | `w_init` | mV |
| $I_{offset}$ | Tonic input current | `i_offset` | `I_offset` | mV/timestep |
| $T_{refract}$ | Refractory period | `T_refract` | `T_refract` | timesteps |
| $I_{syn}$ | Synaptic input current | — (runtime input, not a stored param) | `exc_input + inh_input` (in `neuron_model_state_update`) | mV/timestep |

##### Raw biophysical parameters (Python helper function only — never reach the chip)
| Math symbol | Meaning | Units | Python param | Used for |
|---|---|---|---|---|
| $C_m$ | Membrane capacitance | — | *(not exposed — `tau_m` given directly instead)* | implicitly folds into $\tau_m$ |
| $g_L$ | Leak conductance | nS | `g_L` | rescaling $w$, $a$, $b$, $I$ into mV units |
| $\tau_m$ | Membrane time constant $(=C_m/g_L)$ | ms | `tau_m` | precomputing `alpha_decay` |
| $\tau_w$ | Adaptation time constant | ms | `tau_w` | precomputing `alpha_w` |
| $a$ | Subthreshold coupling | nS | `a` | precomputing `a_sub` |
| $b$ | Spike-triggered adaptation increment | pA | `b` | precomputing `b_inc` |

##### Pre-computed constants (derived in Python helper function, written to `neuron_params_t`)
| Math symbol | Formula | C field | Units |
|---|---|---|---|
| $\alpha_m$ | $\exp(-dt/\tau_m)$ | `alpha_decay` | dimensionless |
| $\alpha_w$ | $\exp(-dt/\tau_w)$ | `alpha_w` | dimensionless |
| $a_{sub}$ | $(a/g_L)\cdot(1-\alpha_w)$ | `a_sub` | mV |
| $b_{inc}$ | $b/g_L$ *(inferred — same rescaling as `w_adapt`; not stated in comments, worth confirming against the Python source)* | `b_inc` | mV |

> **Tip**
>
> **Please note:**
> - **`threshold` vs. `V_peak`**: same quantity, renamed between Python and C
> - **`g_L`, `a`, `tau_m`, `tau_w`, `C_m`**: these parameters are only used in a helper function for pre-computing certain helper variables (see Section 4.4). They never appear in `neuron_params_t` — by design, all division/exponential work involving them is finished before the descriptor reaches the chip.

## 3. Chip-side

As shown in the section before, the AdEx neuron model implementation shares most of its code with the standard LIF neuron implementation for SpiNNaker2. We can therefore start the chip-side implementation by copying the relevant C code files from the `lif_neuron` folders in the default neuron model location at  `py-spinnaker2/src/spinnaker2/libs/Runtime/device/kernels/spiking_neuron_models/` to a new folder, e.g., `lif_adex`. This includes all files (`.c`, `.h`, `CMakeLists.txt`, etc.) from the `lib/lif_neuron` and the `include/lif_neuron` folders, respectively. When following the default neuron model organization, you would therefore also create two new `lif_adex` folders: one in `lib` and a second one in `include`. An overview and explanation of the files is given in the table in [Section 2](#2-process-overview). 

In the following, we separate the necessary implementations on the chip-side for a new neuron model into two categories: the *functional* part and the *parametrization*. The functional part includes all implementations necessary for correctly modeling the neuronal state update, its refractory behavior, and its behavior immediately after a spike has occurred. The parametrization includes general parameters of the neurons and the memory regions the neuron type requires.

> **Tip**
>
> The folder containing the chip-side code (`lif_adex` in our example) does not have to reside within `src/spinnaker2/libs`. You can technically place it wherever you like, even in a separate repository. Depending on the organization of your source files, you might need to update the `CMakeLists.txt` file to set the correct include directories (`INC_DIR`). During the integration in Python, custom neuron folders will be explained further in [Section 4.7](#47-custom-model-directory).

To retrieve the prepared C code snippets we need for creating our custom neuron model, we first load the code to print the individual snippets throughout the next sections:

In [1]:
from helpers.csnippets import CSnippets

csnippets = CSnippets(show_gutter_linenumbers=True)
csnippets.load("./code/neuron_model_adex.c")  # a single file or a directory

CSnippets(6 snippet(s) loaded)

### 3.1 Parametrization

To accomodate the functionality, implemented in the `neuron_model_impl.c` file, we require access to some variables. The state of the neuron is subject to change and is therefore represented by (dynamic) variables in the `neuron_t` class within the header file `neuron_model_impl.h`. The static parameters of the neuron used to update the state variables are, on the other hand, stored in a separate structure - `neuron_params_t`.  

Your custom model needs to define the two data structures. The content of the structures, however, is defined by the individual model, the AdEx model in our case:

In [2]:
csnippets.display("neuron-params")


The state variables of course need to be initialized. This initialization is not performed in the `neuron_model_impl.c` but in the `neuron.c` file. Specifically, the `neuron_initialise()` function clears old spike records and initializes synapses, buffers, and neuron variables: 

In [3]:
csnippets.display("parameter-initialization")


The only change we have to make in this file compared to the standard LIF implementation is to add a definition for the adaptation current `neuron->w_adapt` (the commented line above).

### 3.2 Neuron Model Functionality

The chip-side code updates that are necessary for most models, mainly involve the `neuron_model_impl.c` (.h) file(s). These files include the logic of the neuronal state update (membrane potential). Since this is different for the AdEx model, we need to implement a new function, `_adex_update`, for updating its neuronal state:

In [4]:
csnippets.display("adex-model-update")


Since the refractory behavior of the AdEx model also differs from a regular LIF neuron, we further need to update the state update function that handles the neuronal state update and calls our `_adex_update` function from before:

In [5]:
csnippets.display("model_state_update")


This is particularly important for the AdEx neuron, since it keeps updating its adaptation variable $w$ during the refractory period (see `else` clause above).


Finally, we also need to adapt the function that defines the behavior when the neurons has emitted a spike:

In [6]:
csnippets.display("model_spike")

Compared to the regular LIF model, we only need to add a line of code here for incrementing `w_adapt` by `b_inc`. This facilitates the firing rate dependant adaptation mechanism of the AdEx neuron. 

### 3.3 Memory Management

All of the neuronal parameters, the recorded data, the logging information, etc. are stored on the SRAM of the individual QPE. In order to make sure that we can find the information on-chip that we provide in python during execution, we need to define specific regions, where the data lies in memory. This is done within the `regions.h` file and commonly includes the following regions: 

In [7]:
csnippets.display("memory-regions")


We do not need to change this file at all for the AdEx neuron but can simply copy it from the folder of the basic `lif_neuron`. In case you do need additional regions for your neuron model, you need to define them here, such that the chip knows where to find them. In section **4. Host-side**, we will explain how to define the same structures on the host side in python - a necessary step to write the data into the correct memory regions.

## 4. Host-side

In the previous section we covered all the chip-side code that we need for actually running the instantiation of a neuron model on SpiNNaker2. What is left now, is the host-side code, which is used as a PyNN-like wrapper for creating populations of neurons, connecting them, and running them on-chip. 

In order to run our neuron model in a network on-chip, we need to create a custom Python class for it that inherits from `BaseNeuronApplication` or any other appropriate subclass of it. In this section, we will cover the basic elements of this class and execute them on the fly. When creating a reusable model, you should, however, rather create a separate Python file for this class. This file can have an arbitrary name (with snake_case) and should be located in the `neuron_models` folder, e.g., `py-spinnaker2/src/spinnaker2/neuron_models/lif_adex.py` for our AdEx neuron.

### 4.1 Imports



In [ ]:
from os import PathLike
from pathlib import Path

from collections.abc import Mapping
from typing import Optional

import numpy as np

from spinnaker2.neuron_models.neuron_application import BaseNeuronApplication
from spinnaker2.neuron_models.regions import (
    MemoryContext,
    ResultsContext,
    BaseRegionType,
    GlobalParamsRegion,
    LogRegion,
    NeuronParamsRegion,
    ProfilingRegion,
    RoutingTableRegion,
    SpikesRegion,
    SynapseRegion,
    TimeDoneRegion,
    TimerConfigRegion,
    VoltagesRegion,
)
from spinnaker2.synapses import SynapseWordSize, SynapseWordSpec

### 4.2 Basic Class

Below you can find the code for the core of our custom neuron Python class. To improve the readability of this tutorial, we chose to create some larger attributes of the class in separate cells and subsections. In a regular setup, all of these attributes would be defined within the class itself, as indicated by the commented part of the `LIFAdExApplication` class below. 

The first set of attributes is listed below and part of the class definition in the code cell. 

| Attribute | Description |
|---|---|
| `app_name` | Internal name used by py-spinnaker2 when creating a population object. |
| `dir_name` | Name of the chip-side project folder, only if it is located within the default directory in `src/spinnaker/libs`. |
| `error_log_address` | First available address of DTCM (data region), used for assigning the error log buffer. |
| `first_address` | First usable address of the allocation of memory regions. |
| `reset` | Defines the behavior of the neuron model after emitting a spike, options are [*reset_to_v_reset*, *reset_by_subtraction*] |
| `max_atoms_per_core` | The maximum number of neurons per core (`max_atoms_per_core`). |

The value of the `max_atoms_per_core` variable depends on the complexity of the neuron model, i.e., the number of parameters it requires. In our case, the AdEx model requires much more space for parameters than the regular LIF neuron (56B vs. 24B). We therefore have to reduce the number of neurons per core for this particular neuron type. Currently, there's no automatic way to compute this number, so it must be set empirically when needed. For the AdEx model we chose a reduced value of *128* (*< 256* for LIF), to accommodate for the larger footprint of the neuron model.

Within the neuron model class, we always have to call the constructor (`__init__()`) of the parent class of our custom neuron model to make sure the remaining attributes, memory regions, etc. are correctly initialized. 

In [ ]:
class LIFAdExApplication(BaseNeuronApplication):
    app_name = "lif_adex_new"
    dir_name = "lif_adex_new"

    error_log_address = 0x0000_E000
    first_address = 0x0000_E900

    reset = "reset_to_v_reset"  # AdEx default; or "reset_by_subtraction"
    max_atoms_per_core = 128  # smaller than LIF: params struct is 56B vs 24B

    ## Parameters will be defined separately for better readability
    # sw_spec = ...
    # default_params = ...
    # layout = ...

    def __init__(self):
        super().__init__()

> **Tip**
>
> When you encounter an error during compile time, where the linker complains about overlapping regions, this is fixed most of the time by moving the `error_log_address` further to the back, i.e., increasing the address.

> **Note**
>
> For additional information on the memory structure of the SRAM, please have a look at the [Memory Structure](https://spinnaker2.gitlab.io/py-spinnaker2/advanced_topic/memory/memory-structure.html) documentation page.

### 4.3 Synapse Configuration

By default, most neuron models have synapses with a precision of 16 bit. You can increase the synaptic *word size*, i.e., the overall size in memory for a single synapse, to 32 bit if required. This comes at a cost: by increasing the size of synapses, you limit the remaining memory for neuronal data. Hence, the maximum number of neurons per core (`max_num_atoms_per_core`) might needs to be reduced to fit all synapses with their neurons on the SRAM. 

After defining the overall size of a synapse, you then also need to define the individual size for each of its attributes: `weight`, `delay`, `synapse` and `target`.

All of these attributes' size can be chosen individually but must sum up to the `word_size`.

In [ ]:
sw_spec = SynapseWordSpec(
    word_size=SynapseWordSize.SIZE_16,
    weight=4,
    delay=3,
    synapse_type=1,
    target=8,
)
LIFAdExApplication.sw_spec = sw_spec

### 4.4 Neuronal Parameters

#### Default Neuron Parameters

Each neuron model needs to specify default values for all of its parameters. These parameters are stored as a dictionary within the `default_params` attribute of the neuron model class:

In [ ]:
# Default params — RS (regular spiking) preset, dt=1 ms
# Derived from: tau_m=20ms, tau_w=100ms, a=4nS, g_L=10nS, b=80.5pA
#               E_L=-70mV, V_T=-50mV, Δ_T=2mV, V_reset=-70mV, V_peak=-40mV
default_params = {
    # ── precomputed decay constants ──────────────────────────────────
    "alpha_decay": np.float32(np.exp(-1.0 / 20.0)),  # exp(-dt/τ_m)
    "alpha_w": np.float32(np.exp(-1.0 / 100.0)),  # exp(-dt/τ_w)
    # ── voltage parameters ───────────────────────────────────────────
    "e_l": np.float32(-70.0),  # leak / resting potential
    "threshold": np.float32(-40.0),  # V_peak — hard spike threshold
    "v_t": np.float32(-50.0),  # soft threshold in exp term
    "delta_t": np.float32(2.0),  # slope factor Δ_T (mV)
    # ── adaptation parameters ────────────────────────────────────────
    "a_sub": np.float32(  # (a/g_L)·(1-α_w)
        (4.0 / 10.0) * (1.0 - np.exp(-1.0 / 100.0))
    ),
    "b_inc": np.float32(80.5 / 10.0),  # b_pA / g_L_nS → mV
    # ── standard LIF-compatible fields ──────────────────────────────
    "i_offset": np.float32(0.0),
    "v_init": np.float32(-70.0),
    "w_init": np.float32(0.0),
    "v_reset": np.float32(-70.0),
    "v_rest": np.float32(-70.0),  # for conductance kernels
    # ── int32 — must match last field in neuron_params_t ─────────────
    "T_refract": np.int32(2),
}
LIFAdExApplication.default_params = default_params

> **Info**
>
> You might have noticed that all parameters in the example above have been defined with numpy data types. At the moment, this is **mandatory** to assure the correct encoding before sending it to the chip. Currently, the system supports `np.int32`, `np.uint32` and `np.float32`.

#### Parameter Helper Function

The AdEx neuron model uses several pre-computed parameters to improve the runtime efficiency, as introduced in Section 2. We therefore define a helper functions to aid with the computation of these variables. The `adex_params()` function takes the biophysical parameters from the equations in Section 1 and computes the helper variables:  

In [ ]:
def adex_params(
    tau_m: float = 20.0,  # ms
    tau_w: float = 100.0,  # ms
    a: float = 4.0,  # nS  subthreshold coupling
    g_L: float = 10.0,  # nS  leak conductance
    b: float = 80.5,  # pA  spike-triggered adaptation
    E_L: float = -70.0,  # mV  leak potential
    V_T: float = -50.0,  # mV  exponential soft threshold
    delta_T: float = 2.0,  # mV  slope factor
    V_reset: float = -70.0,  # mV  post-spike reset
    V_peak: float = -40.0,  # mV  hard spike-detection threshold
    V_init: float = -70.0,  # mV  initial membrane voltage
    w_init: float = 0.0,  # mV  initial adaptation current
    i_offset: float = 0.0,  # mV/timestep  tonic input
    T_refract: int = 2,  # timesteps
    dt: float = 1.0,  # ms
    n: int = 1,  # number of neurons (for per-neuron arrays)
) -> dict:
    alpha_decay = float(np.exp(-dt / tau_m))
    alpha_w_val = float(np.exp(-dt / tau_w))

    def _broadcast(v, dtype):
        arr = np.full(n, v, dtype=dtype) if np.isscalar(v) else np.asarray(v, dtype=dtype)
        return arr if len(arr) > 1 else arr[0]

    return {
        "alpha_decay": _broadcast(alpha_decay, np.float32),
        "alpha_w": _broadcast(alpha_w_val, np.float32),
        "e_l": _broadcast(E_L, np.float32),
        "threshold": _broadcast(V_peak, np.float32),
        "v_t": _broadcast(V_T, np.float32),
        "delta_t": _broadcast(delta_T, np.float32),
        "a_sub": _broadcast(
            (a / g_L) * (1.0 - alpha_w_val),
            np.float32,
        ),
        "b_inc": _broadcast(b / g_L, np.float32),
        "i_offset": _broadcast(i_offset, np.float32),
        "v_init": _broadcast(V_init, np.float32),
        "w_init": _broadcast(w_init, np.float32),
        "v_reset": _broadcast(V_reset, np.float32),
        "v_rest": _broadcast(E_L, np.float32),
        "T_refract": _broadcast(T_refract, np.int32),
    }

### 4.5 Memory Layout

Next, we need to define the application's memory `layout` as an ordered list. Each entry must be a class representing a memory region. Standard region classes (e.g., those used by LIFApplication) are defined in `regions.py`. Memory regions defined like this can also have multiple subregions, such as the `SynapseRegion`, which has three subregions.

In [ ]:
layout = [
    RoutingTableRegion,
    TimerConfigRegion,
    GlobalParamsRegion,
    SynapseRegion,
    NeuronParamsRegion,
    SpikesRegion,
    VoltagesRegion,
    TimeDoneRegion,
    LogRegion,
    ProfilingRegion,
]
LIFAdExApplication.layout = layout

> **Warning**
>
> The layout you define here in your Python class and the layout defined in the `regions.h` have to match, i.e., their order has to be the same (the names can differ). Regions that have subregions need to be accounted for as well by including the subregions in the `regions.h` file.
>
> The SynapseRegion, for example, is composed of three subregions, so we include all three in the `regions.h` file:
>
> ```
> POPULATION_TABLE_REGION,        //!< master population table, info; 3
> POPULATION_TABLE_REGION_UNUSED, //!< unused master population table; 4
> SYNAPSE_ROWS_REGION,            //!< synaptic matrix; 5
> ```

> **Note**
>
> For more information on predefined regions refer to [py-spinnaker2 regions-API](https://spinnaker2.gitlab.io/py-spinnaker2/api_reference/regions-api.html).

### 4.6 Custom Memory Regions (Optional)

If your model requires additional regions, you can define them in your model file by inheriting from `BaseRegionType` from `regions.py`, for example:

In [ ]:
class CustomNewRegion(BaseRegionType):
    name = "custom_new_region"
    recordable = True

    @classmethod
    def format_memory_region(cls, ctx: MemoryContext, _pipeline_ctx: Optional[Mapping[str, object]]): ...

    @classmethod
    def format_raw_results(cls, raw_data, ctx: ResultsContext): ...

> **Warning**
>
> Add the new memory region to your `regions.h` as well and remember to keep the same order on both sides.

Standard regions used by LIFApplication can also be overridden as needed for custom models. For example: 

In [ ]:
class CustomGlobalParamsRegion(GlobalParamsRegion):
    @classmethod
    def format_memory_region(cls, ctx: MemoryContext, _pipeline_ctx: Optional[Mapping[str, object]]): ...

### 4.7 Custom Model Directory (Optional)

In case you decided to put your *chip-side* code into a non-standard folder (everywhere, except for the default folder in `src/spinnaker2/libs`), you need to set the path to your model source directory manually. This is done by overwriting the inherited implementation of the `get_app_path()` method within your neuron model class and returning your custom path:

In [ ]:
def get_app_path(self) -> PathLike:
    """Returns the base path for the application binaries."""

    # Original implementation
    #    return backend_settings.get_app_base_path() / self.dir_name

    return Path("/my/custom/path")


LIFAdExApplication.get_app_path = get_app_path

## 5. Model Registration

In order to use your custom neuron model, we need to register it in `py-spinnaker2`. This is only required if you did not place your model in the standard folder (`py-spinnaker2/src/spinnaker2/neuron_models`). The *chip-side* as well as the *host-side* files can reside in arbitrary folders or repositories. They do not have to be located within the `py-spinnaker2` package or any of its submodules. This allows you to store your custom model in separate repositories using the folder structures you prefer:

```markdown
/path/to/your/custom/model/chip-side/
├── CMakeLists.txt
└── ...
/path/to/your/custom/model/host-side/custom_model.py <- contains the class CustomModelApplication>
/path/to/your/custom/model/host-side/example_usage.py
```

The `chip-side` and `host-side` folders are just examples. You could also put everything into a single folder.

For our AdEx example, we follow the standard setup. We choose paths located within the `spinnaker2` package for the *host-side* code and within `src/spinnaker2/libs` for the *chip-side* code. 

```markdown
/py-spinnaker2/src/spinnaker2/libs/Runtime/device/kernels/spiking_neuron_models/
├── lib/
│   └── lif_adex/
│       ├── CMakeLists.txt
│       ├── neuron.c
│       └── ...
└── include/
    └── lif_adex/
        ├── neuron.h
        └── ...
/py-spinnaker2/src/spinnaker2/neuron_models/
└── lif_adex.py   <- contains the class LIFAdExApplication>
```

This means that we technically don't have to register the model in our case. 

We can still run the registration as if it was placed in a custom folder:

In [ ]:
from spinnaker2.neuron_models import register_custom_model

# only needed when implemented as a separate file:
# from lif_adex_new import LIFAdExApplication

register_custom_model(model_class=LIFAdExApplication)

> **Warning**
>
> Please be aware that you currently need to register your model before each run of a script because the registration is only valid for one session.

> **Tip**
>
> If you have placed your *chip-side* files into a custom folder, please remember to set the folder correctly within your custom neuron model *host-side* implementation (see Section 4.7).
>
> In case you choose to place your *host-side* code within a custom folder, you need to make sure that it can be imported during the registration process above. This might require you to add the path to your custom folder to the current `PYTHONPATH`.

## 6. Minimal Running Example

In the previous sections we setup the chip-side and host-side code for our custom AdEx neuron model. Now we want to see it in action! 

To accomplish that, we will create a simple network of an AdEx neuron and input neurons both in simulation (brian2) and on-chip (SpiNNaker2). This will allow us to directly compare the results and see if our implementation works as expected. 

### 6.1 Imports

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from spinnaker2 import hardware, snn

### 6.2 Parameter Configuration

In [ ]:
hw = hardware.SpiNNcloud48NodeBoard()  # or hardware.SpiNNaker2Chip()
record_on_dram = False
model = "lif_adex_new"

timesteps = 200
SEED = 40

In [ ]:
def adex_compute(tau_m, tau_w, a, g_L, b, dt=1.0):
    alpha_decay = float(np.exp(-dt / tau_m))
    alpha_w = float(np.exp(-dt / tau_w))
    a_sub = float((a / g_L) * (1.0 - alpha_w))
    b_inc = float(b / g_L)
    return alpha_decay, alpha_w, a_sub, b_inc


# Three neurons: RS, IB, FS
N = 3

# Physical parameters per neuron (lists, one entry per neuron)
tau_m = [20.0, 5.0, 10.0]  # ms  membrane time constant
tau_w = [100.0, 100.0, 200.0]  # ms  adaptation time constant
a = [4.0, 4.0, 2.0]  # nS  subthreshold coupling
g_L = [10.0, 10.0, 12.0]  # nS  leak conductance
b = [80.5, 0.5, 10.0]  # pA  spike-triggered adaptation increment
E_L = [-70.0, -70.0, -65.0]  # mV  leak/resting potential
V_T = [-50.0, -50.0, -53.0]  # mV  exponential soft threshold
dT = [2.0, 2.0, 1.5]  # mV  slope factor Δ_T
V_reset = [-70.0, -46.0, -65.0]  # mV  post-spike reset (IB: elevated for bursting)
V_peak = [-40.0, -40.0, -45.0]  # mV  hard spike-detection threshold
V_init = [-70.0, -70.0, -65.0]  # mV  initial voltage
T_ref = [2, 2, 2]  # ts  refractory period

# Pre-compute derived quantities per neuron
alpha_decays, alpha_ws, a_subs, b_incs = [], [], [], []
for i in range(N):
    ad, aw, asub, binc = adex_compute(tau_m[i], tau_w[i], a[i], g_L[i], b[i])
    alpha_decays.append(ad)
    alpha_ws.append(aw)
    a_subs.append(asub)
    b_incs.append(binc)

# Build the params dict — field order matches neuron_params_t in neuron_model_impl.h
params = {
    # precomputed decay constants
    "alpha_decay": np.array(alpha_decays, dtype=np.float32),
    "alpha_w": np.array(alpha_ws, dtype=np.float32),
    # voltage parameters
    "e_l": np.array(E_L, dtype=np.float32),
    "threshold": np.array(V_peak, dtype=np.float32),
    "v_t": np.array(V_T, dtype=np.float32),
    "delta_t": np.array(dT, dtype=np.float32),
    # adaptation parameters
    "a_sub": np.array(a_subs, dtype=np.float32),
    "b_inc": np.array(b_incs, dtype=np.float32),
    # standard LIF-compatible
    "i_offset": np.float32(0.0),
    "v_init": np.array(V_init, dtype=np.float32),
    "w_init": np.float32(0.0),
    "v_reset": np.array(V_reset, dtype=np.float32),
    "v_rest": np.array(E_L, dtype=np.float32),
    # refractory (int32)
    "T_refract": np.array(T_ref, dtype=np.int32),
    # global params
    "reset": "reset_to_v_reset",
}

### 6.3 Network Setup

In [ ]:
pop = snn.Population(
    size=N,
    neuron_model=model,
    params=params,
    name="adex_pop",
    record=["spikes", "v"],
)

# Input stimulation
# Shared Poisson background (all neurons) — sustained drive to reveal adaptation
INPUT_SIZE = 50
RATE_HZ = 35
DT = 1
stim_bg = snn.Population(
    size=INPUT_SIZE,
    neuron_model="poisson_process",
    params={"p": RATE_HZ * DT / 1000, "Tref": 3, "seed": 42},
    name="stim_bg",
)

# Deterministic burst to all neurons: spike_list fires at t=5,6,7,8,9
# This triggers the initial response and lets adaptation diverge clearly
burst_times = [5, 6, 7, 8, 9, 50, 51, 52, 53, 54]
stim_burst = snn.Population(
    size=1,
    neuron_model="spike_list",
    params={0: burst_times},
    name="stim_burst",
)

# Connections
# Background Poisson → all 3 AdEx neurons (excitatory, weight=3, delay=1)
bg_conns = [
    [pre, post, 3, 1]
    for pre in range(INPUT_SIZE)
    for post in range(N)
    if np.random.rand() < 0.25  # 25% connection probability
]

# Burst → all 3 neurons (strong excitatory kick, weight=8, delay=0)
burst_conns = [
    [0, 0, 8, 0],  # RS — moderate
    [0, 1, 12, 0],  # IB — stronger kick to trigger burst
    [0, 2, 8, 0],  # FS — moderate
]


proj_bg = snn.Projection(pre=stim_bg, post=pop, connections=bg_conns, name="bg→pop")
proj_burst = snn.Projection(pre=stim_burst, post=pop, connections=burst_conns, name="burst→pop")

# Network
net = snn.Network("adex_example")
net.add(stim_bg, stim_burst, pop, proj_bg, proj_burst)

### 6.4 Run Networks & Retrieve Data

In [ ]:
hw.run(net, timesteps, debug=True)

# Results
spike_times = pop.get_spikes()
voltages = pop.get_voltages()

### 6.5 Plot results

In [ ]:
neuron_labels = [
    "N0 — Regular Spiking\n(τ_w=100ms, b=80.5pA)",
    "N1 — Intrinsic Bursting\n(τ_w=100ms, b=0.5pA)",
    "N2 — Fast Spiking\n(τ_w=200ms, b=10pA)",
]
colors = ["#2471a3", "#c0392b", "#1e8449"]
t_axis = np.arange(timesteps)

fig, axes = plt.subplots(N + 1, 1, figsize=(12, 10), sharex=True)
fig.suptitle("AdEx neuron — three firing modes (SpiNNaker2)", fontsize=13)

# Input burst markers
ax_in = axes[0]
ax_in.eventplot([burst_times], lineoffsets=0, linelengths=0.6, colors="k", linewidths=1.2)
ax_in.set_ylabel("burst\ninput", fontsize=9)
ax_in.set_ylim(-0.5, 0.5)
ax_in.set_yticks([])
ax_in.set_title("Input burst (shared by all neurons)", fontsize=9, loc="left")

# Per-neuron voltage traces + spike markers
for i in range(N):
    ax = axes[i + 1]

    # Voltage trace
    if voltages and i in voltages:
        ax.plot(t_axis, voltages[i], color=colors[i], lw=1.2, label="V_m")
    ax.axhline(V_T[i], ls=":", lw=0.8, color="0.5", label=f"V_T = {V_T[i]} mV")
    ax.axhline(V_peak[i], ls="--", lw=0.8, color="0.3", label=f"V_peak = {V_peak[i]} mV")

    # Spike markers
    spk = spike_times.get(i, [])
    for t in spk:
        ax.axvline(t, color=colors[i], alpha=0.35, lw=0.8)

    n_spikes = len(spk)
    ax.set_ylabel(f"V (mV)\n({n_spikes} spikes)", fontsize=9)
    ax.set_title(neuron_labels[i], fontsize=9, loc="left")
    ax.legend(fontsize=7, loc="upper right", ncol=3)
    ax.set_ylim(min(V_reset) - 5, V_peak[i] + 10)

axes[-1].set_xlabel("Time (ms)")
axes[-1].set_xlim(0, timesteps)

plt.tight_layout()
plt.savefig("adex_output.png", dpi=150)
print("\nFigure saved to adex_output.png")

# Text summary
print("\nSpike times ──────────────────────────────────────────")
for i in range(N):
    spk = spike_times.get(i, [])
    isis = np.diff(spk).tolist() if len(spk) > 1 else []
    print(f"  Neuron {i} ({['RS', 'IB', 'FS'][i]}): {len(spk)} spikes at t={spk}  ISIs={[f'{x:.0f}' for x in isis]} ms")

print("\nAdaptation effect ────────────────────────────────────")
print("  RS (N0): strong b=80.5pA → ISIs lengthen over time (spike-frequency adaptation)")
print("  IB (N1): weak  b=0.5pA  → rapid re-firing, burst-like episodes")
print("  FS (N2): low   b=10pA   → near-constant ISI (minimal adaptation)")
plt.show()

## Recap & what's next

- The AdEx neuron model combines an exponential rise of the membrane potential with an adaptive thresholding based on previous activity.
- Creating custom neuron models involves copying/adapting chip-side code in C and host-side code in Python to enable its use in networks.
- Neuron models can have arbitrary logic and data as long as it fits the framework.
- Custom neuron models can be placed anywhere - even in your own repository - just register your model!

**Coming soon — custom neuron models, advanced**: advancing your understanding of SpiNNaker2 neuron models. 